In [33]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load_dataset import laod_dataframes

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from surprise import SVD, Dataset, Reader

1. Загрузка данных

In [34]:
dataset_dict = laod_dataframes()

df_books = dataset_dict["df_books"]
df_ratings = dataset_dict["df_ratings"]
df_book_tags = dataset_dict["df_book_tags"]
df_tags = dataset_dict["df_tags"]
df_to_read = dataset_dict["df_to_read"]

In [35]:
for name, df in dataset_dict.items():
    print(f"{name}: {df.shape}")
    print(df.columns.tolist())
    print("-" * 50)

df_books: (10000, 23)
['id', 'book_id', 'best_book_id', 'work_id', 'books_count', 'isbn', 'isbn13', 'authors', 'original_publication_year', 'original_title', 'title', 'language_code', 'average_rating', 'ratings_count', 'work_ratings_count', 'work_text_reviews_count', 'ratings_1', 'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5', 'image_url', 'small_image_url']
--------------------------------------------------
df_ratings: (981756, 3)
['book_id', 'user_id', 'rating']
--------------------------------------------------
df_book_tags: (999912, 3)
['goodreads_book_id', 'tag_id', 'count']
--------------------------------------------------
df_tags: (34252, 2)
['tag_id', 'tag_name']
--------------------------------------------------
df_to_read: (912705, 2)
['user_id', 'book_id']
--------------------------------------------------


1.2 Очистка данных

In [ ]:
def clean_ratings(df_ratings, df_books):
    """Отчистка рейтингов"""
    ratings = df_ratings[["user_id", "book_id", "rating"]].copy()

    ratings = ratings.dropna()
    ratings = ratings.groupby(['user_id','book_id' ], as_index=False)['rating'].agg('mean')
    # Оставляем книги, которые присутствуют в каталоге
    ratings = ratings[ratings["book_id"].isin(df_books["id"])]

    return ratings.reset_index(drop=True)


ratings = clean_ratings(df_ratings, df_books)
print("Количество взаимодействий:", len(ratings))
print("Пользователей:", ratings["user_id"].nunique())
print("Книг:", ratings["book_id"].nunique())

1.3. Разделение на train, val и test

In [37]:
import numpy as np

def train_test_val_per_user(df, train_size=0.6, val_size=0.2, test_size=0.2,
                            min_ratings=3, random_state=42):
    """
    Сплит, при котором КАЖДЫЙ пользователь присутствует во всех трёх выборках.
    У каждого пользователя оставляем минимум 1 оценку в train.
    Пользователи с < min_ratings оценками — целиком в train.
    """
    assert abs(train_size + val_size + test_size - 1.0) < 1e-9

    rng = np.random.default_rng(random_state)
    df = df.copy()

    # случайный порядок внутри пользователя
    df["_rand"] = rng.random(len(df))
    df = df.sort_values(["user_id", "_rand"]).reset_index(drop=True)

    df["_rank"] = df.groupby("user_id").cumcount()
    n_per_user = df.groupby("user_id")["_rank"].transform("max") + 1  # Series

    # --- Считаем размеры val/test в numpy, чтобы не ловить pandas-ловушки ---
    n_per_user_np = n_per_user.to_numpy()
    too_few = n_per_user_np < min_ratings

    n_val  = np.maximum(np.floor(n_per_user_np * val_size).astype(int),  1)
    n_test = np.maximum(np.floor(n_per_user_np * test_size).astype(int), 1)

    # если у пользователя слишком мало оценок, чтобы дать всем по 1,
    # уменьшаем val/test, чтобы train остался >= 1
    # (страхуемся на случай min_ratings=3 и n=3: 1/1/1 — ок;
    #  но если n=2 и min_ratings=2 — тоже надо не уйти в 0)
    n_val  = np.where(n_per_user_np - n_val - n_test < 1,
                      np.maximum(n_per_user_np - n_test - 1, 0),
                      n_val)
    n_test = np.where(n_per_user_np - n_val - n_test < 1,
                      np.maximum(n_per_user_np - n_val - 1, 0),
                      n_test)
    n_train = n_per_user_np - n_val - n_test

    # пользователей с < min_ratings целиком в train
    n_train = np.where(too_few, n_per_user_np, n_train)
    n_val   = np.where(too_few, 0, n_val)
    n_test  = np.where(too_few, 0, n_test)

    # ---- привязываем размеры обратно к строкам ----
    # n_per_user — Series с индексом df.index, поэтому map по индексу безопасен
    idx = df.index
    user_vals = df["user_id"].to_numpy()

    # словарь user_id → (n_train, n_val)
    # (проще и надёжнее, чем transform по двум разным колонкам)
    sizes = {}
    seen = {}
    for uid, nt, nv in zip(user_vals, n_train, n_val):
        if uid not in seen:
            seen[uid] = True
            sizes[uid] = (nt, nv)

    df["_n_train"] = df["user_id"].map(lambda u: sizes[u][0])
    df["_n_val"]   = df["user_id"].map(lambda u: sizes[u][1])

    is_train = df["_rank"] < df["_n_train"]
    is_val   = (df["_rank"] >= df["_n_train"]) & (df["_rank"] < df["_n_train"] + df["_n_val"])
    is_test  = ~(is_train | is_val)

    cols_drop = ["_rand", "_rank", "_n_train", "_n_val"]
    train_df = df[is_train].drop(columns=cols_drop).reset_index(drop=True)
    val_df   = df[is_val].drop(columns=cols_drop).reset_index(drop=True)
    test_df  = df[is_test].drop(columns=cols_drop).reset_index(drop=True)

    return train_df, val_df, test_df


train_df, val_df, test_df = train_test_val_per_user(ratings, 0.6, 0.2, 0.2)

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:<6}: {len(d):>8} | users: {d['user_id'].nunique():>6} | books: {d['book_id'].nunique()}")

print("\nusers in train ∩ val :", len(set(train_df['user_id']) & set(val_df['user_id'])))
print("users in train ∩ test:", len(set(train_df['user_id']) & set(test_df['user_id'])))
print("users in val   ∩ test:", len(set(val_df['user_id']) & set(test_df['user_id'])))

train :    56553 | users:  28906 | books: 812
val   :    11489 | users:  10068 | books: 810
test  :    11489 | users:  10068 | books: 812

users in train ∩ val : 10068
users in train ∩ test: 10068
users in val   ∩ test: 10068


1.4 Создаем признаки user_history test_relevant val_relevant

In [38]:
user_history = (
    train_df.groupby("user_id")["book_id"]
    .agg(set)
    .to_dict()
)

# Только оценки 4 и 5 считаем релевантными
test_relevant = (
    test_df[test_df["rating"] >= 4]
    .groupby("user_id")["book_id"]
    .agg(set)
    .to_dict()
)

# Только оценки 4 и 5 считаем релевантными
val_relevant = (
    val_df[val_df["rating"] >= 4]
    .groupby("user_id")["book_id"]
    .agg(set)
    .to_dict()
)

print("Пользователей с релевантными книгами test:",len(test_relevant))
print("Пользователей с релевантными книгами val:",len(val_relevant))

Пользователей с релевантными книгами test: 6952
Пользователей с релевантными книгами val: 6820


1.5. recommend_popular

In [39]:
class recommend_popular_recomemder:
    def __init__(self, train_df, ):
        self.global_mean = train_df["rating"].mean()
        self.m = 50

        self.book_popularity = (
            train_df.groupby("book_id")
            .agg(
                mean_rating=("rating", "mean"),
                ratings_count=("rating", "count")
            )
            .reset_index()
        )

        self.book_popularity["score"] = (
        self.book_popularity["ratings_count"]
        / (self.book_popularity["ratings_count"] + self.m)
        * self.book_popularity["mean_rating"]
        +
        self.m / (self.book_popularity["ratings_count"] + self.m)
        * self.global_mean
    )
        self.book_popularity = self.book_popularity.sort_values(
            "score", ascending=False, kind="stable"
        )


    def recommend(self, user_id, N=10):
        seen = user_history.get(user_id, set())

        candidates = self.book_popularity[
            ~self.book_popularity["book_id"].isin(seen)
        ].head(N)

        return [
            {
                "book_id": int(row.book_id),
                "score": float(row.score)
            }
            for row in candidates.itertuples()
    ]


In [40]:
popularity_model = recommend_popular_recomemder(train_df)
test_user = 1
print(f'Рекомендации для пользовател {test_user}')
print(popularity_model.recommend(test_user,10))

Рекомендации для пользовател 1
[{'book_id': 1, 'score': 4.074375307923379}, {'book_id': 2, 'score': 4.009795651005737}, {'book_id': 3, 'score': 3.56345454074686}, {'book_id': 5, 'score': 3.89567463652323}, {'book_id': 6, 'score': 4.0008983912911615}, {'book_id': 8, 'score': 3.768811683604002}, {'book_id': 10, 'score': 4.06345454074686}, {'book_id': 11, 'score': 4.066999244327672}, {'book_id': 13, 'score': 4.009795651005737}, {'book_id': 21, 'score': 4.0444079005534626}]


1.6 Content_based рекомендации

In [ ]:
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer

class content_based_recomender:
    def __init__(self, train_df, df_book_tags, df_books):
        pass

        # 1. Объединяем теги с их названиями
        self.book_tags = df_book_tags.merge(
            df_tags,
            on="tag_id",
            how="left"
        )

        # 2. Выбираем 10 наиболее значимых тегов для каждой книги
        self.book_tags = (
            self.book_tags
            .sort_values("count", ascending=False)
            .groupby("goodreads_book_id")
            .head(10)
            .copy()
        )

        # 3. Подготавливаем названия тегов
        self.book_tags["tag_name"] = (
            self.book_tags["tag_name"]
            .fillna("")
            .str.replace("-", " ", regex=False)
        )

        # 4. Объединяем теги в одну строку для каждой книги
        self.tags_aggregated = (
            self.book_tags.groupby("goodreads_book_id")["tag_name"]
            .agg(" ".join)
            .reset_index()
        )

        # 5. Создаём таблицу с текстовым описанием книг
        self.books_content = df_books[
            ["id", "book_id", "title", "original_title"]
        ].copy()

        # 6. Объединяем книги с тегами по разным названиям ключа
        self.books_content = self.books_content.merge(
            self.tags_aggregated,
            left_on="book_id",
            right_on="goodreads_book_id",
            how="left"
        )

        # 7. Формируем текст для TF-IDF
        self.books_content["content"] = (
            self.books_content["original_title"]
            .fillna(self.books_content["title"])
            .fillna("")
            + " "
            + self.books_content["tag_name"].fillna("")
        )


        self.tfidf = TfidfVectorizer(
            stop_words="english",
            max_features=20000,
            ngram_range=(1, 2)
        )

        self.tfidf_matrix = self.tfidf.fit_transform(
            self.books_content["content"]
        )

        # TfidfVectorizer уже нормализует строки по L2.
        self.tfidf_transpose = self.tfidf_matrix.T.tocsr()

        self.book_id_to_idx = {
            int(book_id): idx
            for idx, book_id in enumerate(self.books_content["id"])
        }

        self.positive_history = (
            train_df[train_df["rating"] >= 4]
            .groupby("user_id")["book_id"]
            .agg(list)
            .to_dict()
        )


        self.global_mean = train_df["rating"].mean()
        self.m = 50

        self.book_popularity = (
            train_df.groupby("book_id")
            .agg(
                mean_rating=("rating", "mean"),
                ratings_count=("rating", "count")
            )
            .reset_index()
        )

        self.book_popularity["score"] = (
        self.book_popularity["ratings_count"]
        / (self.book_popularity["ratings_count"] + self.m)
        * self.book_popularity["mean_rating"]
        +
        self.m / (self.book_popularity["ratings_count"] + self.m)
        * self.global_mean
    )
        self.book_popularity = self.book_popularity.sort_values(
            "score", ascending=False, kind="stable"
        )


    def recommend_from_top(self, user_id, N=10):
        seen = user_history.get(user_id, set())

        candidates = self.book_popularity[
            ~self.book_popularity["book_id"].isin(seen)
        ].head(N)

        return [
            {
                "book_id": int(row.book_id),
                "score": float(row.score)
            }
            for row in candidates.itertuples()
    ]



    def recommend(self, user_id, N=10):
        liked_books = self.positive_history.get(user_id, [])

        indices = [
            self.book_id_to_idx[book_id]
            for book_id in liked_books
            if book_id in self.book_id_to_idx
        ]

        if not indices:
            return self.recommend_from_top(user_id, N)

        user_profile = np.asarray(
            self.tfidf_matrix[indices].mean(axis=0)
        )

        user_profile = normalize(user_profile)

        scores = (csr_matrix(user_profile) @ self.tfidf_transpose).toarray().ravel()

        seen = user_history.get(user_id, set())

        for book_id in seen:
            idx = self.book_id_to_idx.get(book_id)
            if idx is not None:
                scores[idx] = -np.inf

        top_indices = np.argsort(scores)[::-1][:N]

        return [
            {
                "book_id": int(self.books_content.iloc[idx]["id"]),
                "score": float(scores[idx])
            }
            for idx in top_indices
            if np.isfinite(scores[idx])
        ]

In [42]:
content_model = content_based_recomender( train_df, df_book_tags, df_books)
test_user = 1
print(f'Рекомендации для пользовател {test_user}')
print(content_model.recommend(test_user,10))

Рекомендации для пользовател 1
[{'book_id': 1, 'score': 4.074375307923379}, {'book_id': 2, 'score': 4.009795651005737}, {'book_id': 3, 'score': 3.56345454074686}, {'book_id': 5, 'score': 3.89567463652323}, {'book_id': 6, 'score': 4.0008983912911615}, {'book_id': 8, 'score': 3.768811683604002}, {'book_id': 10, 'score': 4.06345454074686}, {'book_id': 11, 'score': 4.066999244327672}, {'book_id': 13, 'score': 4.009795651005737}, {'book_id': 21, 'score': 4.0444079005534626}]


1.7 Коллаборативная фильтрация

In [ ]:


from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

class cb_item_recomennder:
    def __init__(self, train_df, df_books):
        self.user_ids = np.sort(train_df["user_id"].unique())
        self.book_ids = np.sort(df_books["id"].unique())

        self.user_to_idx = {
            user_id: idx
            for idx, user_id in enumerate(self.user_ids)
        }

        self.book_to_idx = {
            book_id: idx
            for idx, book_id in enumerate(self.book_ids)
        }

        self.rows = train_df["user_id"].map(self.user_to_idx).to_numpy()
        self.cols = train_df["book_id"].map(self.book_to_idx).to_numpy()

        self.interaction_matrix = csr_matrix(
            (
                train_df["rating"].to_numpy(dtype=np.float32),
                (self.rows, self.cols)
            ),
            shape=(len(self.user_ids), len(self.book_ids)),
            dtype=np.float32
        )


        self.item_matrix = self.interaction_matrix.T.tocsr()

        self.item_similarity = cosine_similarity(
            self.item_matrix,
            dense_output=False
        )

        self.item_similarity.setdiag(0)
        self.item_similarity.eliminate_zeros()


        self.global_mean = train_df["rating"].mean()
        self.m = 50

        self.book_popularity = (
            train_df.groupby("book_id")
            .agg(
                mean_rating=("rating", "mean"),
                ratings_count=("rating", "count")
            )
            .reset_index()
        )

        self.book_popularity["score"] = (
        self.book_popularity["ratings_count"]
        / (self.book_popularity["ratings_count"] + self.m)
        * self.book_popularity["mean_rating"]
        +
        self.m / (self.book_popularity["ratings_count"] + self.m)
        * self.global_mean
    )
        self.book_popularity = self.book_popularity.sort_values(
            "score", ascending=False, kind="stable"
        )


    def recommend_from_top(self, user_id, N=10):
        seen = user_history.get(user_id, set())

        candidates = self.book_popularity[
            ~self.book_popularity["book_id"].isin(seen)
        ].head(N)

        return [
            {
                "book_id": int(row.book_id),
                "score": float(row.score)
            }
            for row in candidates.itertuples()
    ]



    def recommend(self,user_id, N=10):
        if N <= 0:
            return []
        if user_id not in self.user_to_idx:
            return self.recommend_from_top(user_id, N)

        user_idx = self.user_to_idx[user_id]
        user_ratings = self.interaction_matrix.getrow(user_idx)
        # Матрица cosine similarity симметрична: обходим только оценки пользователя.
        weighted_scores = (user_ratings @ self.item_similarity).toarray().ravel()

        observed = user_ratings.copy()
        observed.data = np.ones_like(observed.data)
        normalizer = (observed @ self.item_similarity).toarray().ravel()
        scores = np.divide(
            weighted_scores,
            normalizer,
            out=np.zeros_like(weighted_scores),
            where=normalizer > 0,
        )
        scores[normalizer <= 0] = -np.inf
        scores[user_ratings.indices] = -np.inf

        valid_indices = np.where(np.isfinite(scores))[0]
        top_indices = valid_indices[
            np.argsort(scores[valid_indices])[::-1][:N]
        ]
        return [
            {"book_id": int(self.book_ids[idx]), "score": float(scores[idx])}
            for idx in top_indices
        ]


In [44]:
item_cf_model = cb_item_recomennder( train_df, df_books)
test_user = 1
print(f'Рекомендации для пользовател {test_user}')
print(item_cf_model.recommend(test_user,10))

Рекомендации для пользовател 1
[{'book_id': 1, 'score': 4.074375307923379}, {'book_id': 2, 'score': 4.009795651005737}, {'book_id': 3, 'score': 3.56345454074686}, {'book_id': 5, 'score': 3.89567463652323}, {'book_id': 6, 'score': 4.0008983912911615}, {'book_id': 8, 'score': 3.768811683604002}, {'book_id': 10, 'score': 4.06345454074686}, {'book_id': 11, 'score': 4.066999244327672}, {'book_id': 13, 'score': 4.009795651005737}, {'book_id': 21, 'score': 4.0444079005534626}]


In [ ]:
from surprise import SVD, Dataset, Reader


class svd_recommender:
    def __init__(self, train_df)  -> None:
        pass
        self.reader = Reader(rating_scale=(1, 5))

        dataset = Dataset.load_from_df(
            train_df[["user_id", "book_id", "rating"]],
            self.reader
        )

        self.trainset = dataset.build_full_trainset()

        self.svd_model = SVD(
            n_factors=50,
            n_epochs=50,
            lr_all=0.005,
            reg_all=0.02,
            random_state=42
        )

        self.svd_model.fit(self.trainset)

        self.user_ids = np.sort(train_df["user_id"].unique())
        self.book_ids = np.sort(df_books["id"].unique())

        self.user_history = (
        train_df.groupby("user_id")["book_id"]
        .agg(set)
        .to_dict()
)

        # Выровненные по каталогу факторы; неизвестные книги имеют нулевые факторы.
        self.catalog_factors = np.zeros((len(self.book_ids), self.svd_model.n_factors))
        self.catalog_bias = np.zeros(len(self.book_ids))
        book_to_idx = {int(b): i for i, b in enumerate(self.book_ids)}
        for inner in self.trainset.all_items():
            idx = book_to_idx[int(self.trainset.to_raw_iid(inner))]
            self.catalog_factors[idx] = self.svd_model.qi[inner]
            self.catalog_bias[idx] = self.svd_model.bi[inner]
        self.user_to_inner = {self.trainset.to_raw_uid(i): i
                              for i in self.trainset.all_users()}

    def recommend(self, user_id, N=10):
        if N <= 0:
            return []
        scores = self.trainset.global_mean + self.catalog_bias
        inner = self.user_to_inner.get(user_id)
        if inner is not None:
            # Та же формула Surprise SVD, сразу для всего каталога.
            scores = scores + self.svd_model.bu[inner] + np.einsum(
                "ij,j->i", self.catalog_factors, self.svd_model.pu[inner]
            )
        scores = np.clip(scores, *self.trainset.rating_scale)
        seen = self.user_history.get(user_id, set())
        scores[np.isin(self.book_ids, list(seen))] = -np.inf
        valid = np.flatnonzero(np.isfinite(scores))
        top = valid[np.argsort(-scores[valid], kind="stable")[:N]]
        return [{"book_id": int(self.book_ids[i]), "score": float(scores[i])} for i in top]


svd_model = svd_recommender(train_df)
test_user = 1
print(f'Рекомендации для пользовател {test_user}')
print(svd_model.recommend(test_user,10))

Метрики

In [46]:
import numpy as np

def precision_at_k(actual, predicted, k=10):
    if k <= 0:
        return 0.0

    actual = set(actual)
    predicted = list(dict.fromkeys(predicted))[:k]

    hits = len(actual.intersection(predicted))

    return hits / k


def recall_at_k(actual, predicted, k=10):
    actual = set(actual)

    if not actual or k <= 0:
        return 0.0

    predicted = list(dict.fromkeys(predicted))[:k]

    hits = len(actual.intersection(predicted))

    return hits / len(actual)


def ndcg_at_k(actual, predicted, k=10):
    actual = set(actual)

    if not actual or k <= 0:
        return 0.0

    predicted = list(dict.fromkeys(predicted))[:k]

    relevance = np.array([
        1 if book_id in actual else 0
        for book_id in predicted
    ])

    positions = np.arange(1, len(relevance) + 1)

    dcg = np.sum(
        relevance / np.log2(positions + 1)
    )

    ideal_hits = min(len(actual), k)

    ideal_positions = np.arange(1, ideal_hits + 1)

    idcg = np.sum(
        1 / np.log2(ideal_positions + 1)
    )

    return float(dcg / idcg)

def evaluate_model(recommend_fn, relevant, k=10, verbose=False):
    from time import perf_counter

    results = []
    started = perf_counter()

    for index, (user_id, actual) in enumerate(relevant.items(), start=1):
        raw_predicted = recommend_fn(user_id, N=k)
        predicted = [rec["book_id"] for rec in raw_predicted]

        results.append({
            "user_id": user_id,
            "precision": precision_at_k(actual, predicted, k),
            "recall": recall_at_k(actual, predicted, k),
            "ndcg": ndcg_at_k(actual, predicted, k),
        })

        if verbose and (index % 500 == 0 or index == len(relevant)):
            elapsed = perf_counter() - started
            eta = elapsed / index * (len(relevant) - index)
            print(f"{index}/{len(relevant)} пользователей | {elapsed:.1f} с | осталось ≈ {eta:.1f} с", flush=True)

    return pd.DataFrame(
        results,
        columns=["user_id", "precision", "recall", "ndcg"],
    )

In [ ]:
models = {
    "Popularity": popularity_model.recommend,
    "Content-Based": content_model.recommend,
    "Item-Based CF": item_cf_model.recommend,
    "SVD": svd_model.recommend,
}


from time import perf_counter
from tqdm.auto import tqdm

def evaluate_model(recommend_fn, relevant, k=10):
    results = []
    started = perf_counter()

    for user_id, actual in tqdm(
        relevant.items(),
        total=len(relevant),
        desc=type(recommend_fn.__self__).__name__,
    ):
        raw_predicted = recommend_fn(user_id, N=k)
        predicted = [rec["book_id"] for rec in raw_predicted]

        results.append({
            "user_id": user_id,
            "precision": precision_at_k(actual, predicted, k),
            "recall": recall_at_k(actual, predicted, k),
            "ndcg": ndcg_at_k(actual, predicted, k),
        })

    print(f"Затрачено: {perf_counter() - started:.1f} с")

    return pd.DataFrame(
        results,
        columns=["user_id", "precision", "recall", "ndcg"],
    )


# Базовые модели обучены на train_df; сравниваем их на val.
# Итоговая оценка всех моделей на test находится после подбора весов.
evaluation_relevant = val_relevant

val_comparison_results = []

for model_name, recommend_fn in models.items():
    print(f"Оценка {model_name}: {len(evaluation_relevant)} пользователей", flush=True)
    metrics = evaluate_model(
        recommend_fn,
        relevant=evaluation_relevant,
        k=10,
        verbose=True,
    )

    val_comparison_results.append({
        "Model": model_name,
        "Precision@10": metrics["precision"].mean(),
        "Recall@10": metrics["recall"].mean(),
        "nDCG@10": metrics["ndcg"].mean(),
    })

val_comparison_df = pd.DataFrame(val_comparison_results)

display(
    val_comparison_df
    .sort_values("nDCG@10", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
import matplotlib.pyplot as plt

val_comparison_df.set_index("Model")[
    ["Precision@10", "Recall@10", "nDCG@10"]
].plot(
    kind="bar",
    figsize=(11, 5),
    rot=0
)

plt.title("Сравнение базовых моделей на val")
plt.xlabel("Модель")
plt.ylabel("Значение метрики")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

Этап 2. Построение гибридной системы

Стратегия объединения моделей:
Взвешенное усреднение предсказаний разных моделей.
Разделение пользователей по типам (новые/активные).
Комбинирование персонализированных и популярных рекомендаций.
Система генерации кандидатов:
Объединение рекомендаций от всех моделей в общий пул.
Балансировка между разнообразием и релевантностью.
Фильтрация уже прочитанных книг.

In [ ]:
import numpy as np
import pandas as pd

def normalize_scores(recommendations):
    if not recommendations:
        return {}

    df = pd.DataFrame(recommendations)

    min_score = df["score"].min()
    max_score = df["score"].max()

    if np.isclose(min_score, max_score):
        df["normalized_score"] = 1.0
    else:
        df["normalized_score"] = (
            (df["score"] - min_score)
            / (max_score - min_score)
        )

    return dict(zip(
        df["book_id"],
        df["normalized_score"]
    ))

In [57]:
DEFAULT_WEIGHTS = {
    "pop": 0.1,
    "content": 0.2,
    "item": 0.3,
    "svd": 0.4,
}


class HybridRecommender:
    def __init__(self, train_df):
        # Инициализируем базовые модели
        self.models = {
            "pop": recommend_popular_recomemder(train_df),
            "content": content_based_recomender( train_df, df_book_tags, df_books),
            "item": cb_item_recomennder( train_df, df_books),
            "svd": svd_recommender(train_df)
        }

        # История пользователей только из обучающей выборки
        self.user_history = (
            train_df.groupby("user_id")["book_id"]
            .agg(set)
            .to_dict()
        )

    def normalize_scores(self, recommendations):
        if not recommendations:
            return {}

        df = pd.DataFrame(recommendations)

        min_score = df["score"].min()
        max_score = df["score"].max()

        if np.isclose(min_score, max_score):
            df["normalized_score"] = 1.0
        else:
            df["normalized_score"] = (
                (df["score"] - min_score)
                / (max_score - min_score)
            )

        return dict(zip(
            df["book_id"],
            df["normalized_score"]
        ))

    def recommend(self, user_id, N=10, weights=None, candidates_k=100):
        weights = dict(
            DEFAULT_WEIGHTS if weights is None else weights
        )

        if set(weights) != set(self.models):
            raise ValueError(
                "Веса должны содержать ключи: pop, content, item, svd"
            )

        if any(not np.isfinite(w) or w < 0 for w in weights.values()):
            raise ValueError(
                "Веса должны быть конечными неотрицательными числами"
            )

        if not np.isclose(sum(weights.values()), 1.0):
            raise ValueError("Сумма весов должна равняться 1")

        # Получаем рекомендации базовых моделей
        recommendations = {
            name: model.recommend(user_id, N=candidates_k)
            for name, model in self.models.items()
        }

        # Используем существующую функцию normalize_scores:
        # она должна возвращать словарь {book_id: normalized_score}
        normalized = {
            name: self.normalize_scores(recs)
            for name, recs in recommendations.items()
        }

        candidate_ids = set().union(
            *(scores.keys() for scores in normalized.values())
        )

        # Исключаем прочитанные книги
        seen = self.user_history.get(user_id, set())
        candidate_ids -= seen

        results = []

        for book_id in candidate_ids:
            score = sum(
                weights[name] * normalized[name].get(book_id, 0.0)
                for name in self.models
            )

            results.append({
                "book_id": int(book_id),
                "score": float(score),
            })

        results.sort(key=lambda rec: (-rec["score"], rec["book_id"]))

        return results[:N]

In [58]:
hybrid_model = HybridRecommender(train_df)

hybrid_recs = hybrid_model.recommend(user_id=1, N=10)
display(pd.DataFrame(hybrid_recs))

,book_id,score
0,976,0.743939
1,870,0.703192
2,998,0.643769
3,25,0.627701
4,760,0.603199
5,291,0.575568
6,1,0.546726
7,250,0.540221
8,365,0.524576
9,50,0.497610


Адаптивная модель

In [59]:
ADAPTIVE_WEIGHTS = {
    "new": {
        "pop": 0.65,
        "content": 0.25,
        "item": 0.05,
        "svd": 0.05,
    },
    "regular": {
        "pop": 0.15,
        "content": 0.35,
        "item": 0.25,
        "svd": 0.25,
    },
    "active": {
        "pop": 0.05,
        "content": 0.15,
        "item": 0.35,
        "svd": 0.45,
    },
}


class AdaptiveHybridRecommender(HybridRecommender):
    def __init__(self, train_df, adaptive_weights=None):
        # Инициализируем базовые модели и историю пользователей
        super().__init__(train_df)

        self.user_activity = (
            train_df.groupby("user_id")
            .size()
            .to_dict()
        )

        source_weights = (
            ADAPTIVE_WEIGHTS
            if adaptive_weights is None
            else adaptive_weights
        )

        self.adaptive_weights = {
            user_type: dict(weights)
            for user_type, weights in source_weights.items()
        }

    def get_user_type(self, user_id):
        n_ratings = self.user_activity.get(user_id, 0)

        if n_ratings < 5:
            return "new"
        elif n_ratings < 50:
            return "regular"
        else:
            return "active"

    def recommend(self, user_id, N=10, weights=None, candidates_k=100):
        # Явно переданные веса имеют приоритет
        if weights is None:
            user_type = self.get_user_type(user_id)
            weights = self.adaptive_weights[user_type]

        # Объединение рекомендаций и проверку весов выполняет родитель
        return super().recommend(
            user_id=user_id,
            N=N,
            weights=weights,
            candidates_k=candidates_k,
        )

In [60]:
adaprive_hybrid_model = AdaptiveHybridRecommender(train_df)

adaptive_recs = adaprive_hybrid_model.recommend(user_id=1, N=10)
display(pd.DataFrame(hybrid_recs))

,book_id,score
0,976,0.743939
1,870,0.703192
2,998,0.643769
3,25,0.627701
4,760,0.603199
5,291,0.575568
6,1,0.546726
7,250,0.540221
8,365,0.524576
9,50,0.497610


In [ ]:
import optuna


def optimize_adaptive_weights(model, val_relevant, n_trials=30, k=10,
                              candidates_k=100, random_state=42):
    if n_trials < 1:
        raise ValueError("n_trials должен быть положительным")
    best_weights = {group: dict(w) for group, w in model.adaptive_weights.items()}
    results = []
    names = list(model.models)

    for user_type in ("new", "regular", "active"):
        relevant = {u: actual for u, actual in val_relevant.items()
                    if model.get_user_type(u) == user_type}
        if not relevant:
            continue  # Для пустой группы сохраняем исходные веса.

        # Рекомендации базовых моделей считаем один раз, переиспользуем в trials.
        cache = {}
        for user_id in relevant:
            normalized = [model.normalize_scores(base.recommend(user_id, N=candidates_k))
                          for base in model.models.values()]
            ids = sorted(set().union(*(scores.keys() for scores in normalized))
                         - model.user_history.get(user_id, set()))
            matrix = np.array([[scores.get(book_id, 0.0) for scores in normalized]
                               for book_id in ids], dtype=float).reshape(-1, len(names))
            cache[user_id] = (np.asarray(ids, dtype=int), matrix)

        def objective(trial):
            raw = np.array([trial.suggest_float(name, 1e-6, 1.0) for name in names])
            weights = raw / raw.sum()
            scores = []
            for user_id, actual in relevant.items():
                ids, matrix = cache[user_id]
                order = np.lexsort((ids, -(matrix @ weights)))[:k]
                scores.append(ndcg_at_k(actual, ids[order], k))
            score = float(np.mean(scores))
            trial.set_user_attr("weights", dict(zip(names, weights.tolist())))
            results.append({"user_type": user_type, "trial": trial.number,
                            **trial.user_attrs["weights"], f"nDCG@{k}": score})
            return score

        study = optuna.create_study(
            direction="maximize", sampler=optuna.samplers.TPESampler(seed=random_state)
        )
        # Исходные веса тоже участвуют: результат не хуже исходного на val.
        study.enqueue_trial({name: max(best_weights[user_type][name], 1e-6)
                             for name in names})
        study.optimize(objective, n_trials=n_trials)
        best_weights[user_type] = study.best_trial.user_attrs["weights"]

    return best_weights, pd.DataFrame(results)


In [ ]:
# Создаём модель и инициализируем базовые модели
adaptive_model = AdaptiveHybridRecommender(train_df)

# Подбираем веса на валидационной выборке
best_weights, search_results = optimize_adaptive_weights(
    model=adaptive_model,
    val_relevant=val_relevant,
    n_trials=30,
)

# Применяем найденные веса
adaptive_model.adaptive_weights = best_weights

display(pd.DataFrame(best_weights).T)

# Получаем рекомендации с подобранными весами
display(pd.DataFrame(
    adaptive_model.recommend(user_id=1, N=10)
))

Two Tower: две независимые башни пользователя и книги

Каждая башня: ID embedding → Linear → ReLU → Linear → нормализация.
Обучение — BPR loss на оценках ≥ 4 из train и случайных книгах, отсутствующих в истории train. Лучшее состояние выбираем по nDCG@10 на val; test при обучении не используется.
Непрочитанные пользователем книги ранжируем по скалярному произведению векторов. Для неизвестного пользователя используем популярность из train; книги без обучающих взаимодействий не ранжируем случайными эмбеддингами. `df_to_read` не используем: у него нет разбиения train/val/test.


In [ ]:
import copy
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset


class TwoTower(nn.Module):
    def __init__(self, n_users, n_books, embedding_dim=64, output_dim=32):
        super().__init__()
        def tower(size):
            return nn.Sequential(nn.Embedding(size, embedding_dim),
                                 nn.Linear(embedding_dim, embedding_dim), nn.ReLU(),
                                 nn.Linear(embedding_dim, output_dim))
        self.user_tower = tower(n_users)
        self.book_tower = tower(n_books)

    def forward(self, users, books):
        u = F.normalize(self.user_tower(users), dim=-1)
        b = F.normalize(self.book_tower(books), dim=-1)
        return (u * b).sum(dim=-1)


class TwoTowerRecommender:
    def __init__(self, train_df, df_books, embedding_dim=64, output_dim=32,
                 random_state=42, device=None):
        torch.manual_seed(random_state)
        self.rng = np.random.default_rng(random_state)
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.user_ids = np.sort(train_df["user_id"].unique())
        self.book_ids = np.sort(df_books["id"].unique())
        self.user_to_idx = {u: i for i, u in enumerate(self.user_ids)}
        self.book_to_idx = {b: i for i, b in enumerate(self.book_ids)}
        self.user_history = train_df.groupby("user_id")["book_id"].agg(set).to_dict()
        self.known_books = np.isin(self.book_ids, train_df["book_id"].unique())
        self.popular = train_df.groupby("book_id").size().sort_values(ascending=False)
        users = train_df["user_id"].map(self.user_to_idx).to_numpy(dtype=np.int64)
        books = train_df["book_id"].map(self.book_to_idx).to_numpy(dtype=np.int64)
        self.seen_keys = np.unique(users * len(self.book_ids) + books)
        can_sample = train_df["user_id"].map(
            {u: len(seen) < len(self.book_ids) for u, seen in self.user_history.items()}
        ).to_numpy()
        positive = (train_df["rating"].to_numpy() >= 4) & can_sample
        self.pairs = np.column_stack((users[positive], books[positive]))
        if not len(self.pairs):
            raise ValueError("Нужны положительные train-пары и непрочитанные книги")
        self.network = TwoTower(len(self.user_ids), len(self.book_ids),
                                embedding_dim, output_dim).to(self.device)
        self._cache_vectors()

    @torch.no_grad()
    def _cache_vectors(self):
        self.network.eval()
        self.user_vectors = F.normalize(self.network.user_tower(
            torch.arange(len(self.user_ids), device=self.device)), dim=-1).cpu().numpy()
        self.book_vectors = F.normalize(self.network.book_tower(
            torch.arange(len(self.book_ids), device=self.device)), dim=-1).cpu().numpy()

    def fit(self, val_relevant, epochs=5, batch_size=1024, lr=1e-3, k=10):
        if not val_relevant or epochs < 1:
            raise ValueError("Нужны непустая val и epochs >= 1")
        optimizer = torch.optim.Adam(self.network.parameters(), lr=lr, weight_decay=1e-5)
        generator = torch.Generator().manual_seed(42)
        best_score, best_state, history = -np.inf, None, []
        for epoch in range(1, epochs + 1):
            # Исключаем ВСЮ историю train из отрицательных примеров, включая низкие оценки.
            negatives = self.rng.integers(len(self.book_ids), size=len(self.pairs))
            invalid = np.isin(self.pairs[:, 0] * len(self.book_ids) + negatives, self.seen_keys)
            while invalid.any():
                negatives[invalid] = self.rng.integers(len(self.book_ids), size=invalid.sum())
                invalid = np.isin(self.pairs[:, 0] * len(self.book_ids) + negatives, self.seen_keys)
            loader = DataLoader(TensorDataset(torch.from_numpy(self.pairs[:, 0].copy()),
                                             torch.from_numpy(self.pairs[:, 1].copy()),
                                             torch.from_numpy(negatives)),
                                batch_size=batch_size, shuffle=True, generator=generator)
            self.network.train()
            total_loss = 0.0
            for users, positive, negative in loader:
                users, positive, negative = [x.to(self.device) for x in (users, positive, negative)]
                loss = F.softplus((self.network(users, negative)
                                   - self.network(users, positive)) / 0.1).mean()
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                total_loss += loss.item() * len(users)
            self._cache_vectors()
            score = evaluate_model(self.recommend, val_relevant, k=k)["ndcg"].mean()
            history.append({"epoch": epoch, "loss": total_loss / len(self.pairs),
                            f"val_nDCG@{k}": score})
            if score > best_score:
                best_score = score
                self.best_epoch = epoch
                best_state = copy.deepcopy(self.network.state_dict())
            print(f"Epoch {epoch}: loss={history[-1]['loss']:.4f}, val nDCG@{k}={score:.4f}")
        self.network.load_state_dict(best_state)
        self._cache_vectors()
        self.history = pd.DataFrame(history)
        return self

    def recommend(self, user_id, N=10):
        if N <= 0:
            return []
        seen = self.user_history.get(user_id, set())
        if user_id not in self.user_to_idx:
            return [{"book_id": int(b), "score": float(count)}
                    for b, count in self.popular.items() if b not in seen][:N]
        scores = self.book_vectors @ self.user_vectors[self.user_to_idx[user_id]]
        scores[~self.known_books | np.isin(self.book_ids, list(seen))] = -np.inf
        valid = np.flatnonzero(np.isfinite(scores))
        top = valid[np.lexsort((self.book_ids[valid], -scores[valid]))[:N]]
        return [{"book_id": int(self.book_ids[i]), "score": float(scores[i])} for i in top]


In [ ]:
two_tower_model = TwoTowerRecommender(train_df, df_books, random_state=RANDOM_STATE)
two_tower_model.fit(val_relevant, epochs=5, batch_size=1024)
display(two_tower_model.history)
print("Лучшая эпоха по val:", two_tower_model.best_epoch)
display(pd.DataFrame(two_tower_model.recommend(user_id=1, N=10)))


Итоговое сравнение всех моделей на test

Все модели обучены на `train_df`. Выбор моделей и подбор адаптивных весов выполняются на `val_relevant`; найденные веса уже применены к `adaptive_model`.
Лучшая эпоха Two Tower также выбрана только на val.
На `test_relevant` считаем итоговые Precision@10, Recall@10 и nDCG@10 для всех моделей без дополнительного подбора. Учитываем всех пользователей с релевантными книгами (оценка ≥ 4) в test.


In [ ]:
test_models = {
    **models,
    "Two Tower": two_tower_model.recommend,
    "Hybrid": hybrid_model.recommend,
    "Adaptive Hybrid": adaprive_hybrid_model.recommend,
    "Adaptive Hybrid (tuned)": adaptive_model.recommend,
}

comparison_results = []
test_metrics_by_model = {}

for model_name, recommend_fn in test_models.items():
    print(f"Оценка {model_name}: {len(test_relevant)} пользователей", flush=True)
    metrics = evaluate_model(
        recommend_fn,
        relevant=test_relevant,
        k=10,
        verbose=True,
    )
    test_metrics_by_model[model_name] = metrics

    comparison_results.append({
        "Model": model_name,
        "Precision@10": metrics["precision"].mean(),
        "Recall@10": metrics["recall"].mean(),
        "nDCG@10": metrics["ndcg"].mean(),
    })

comparison_df = (
    pd.DataFrame(comparison_results)
    .sort_values("nDCG@10", ascending=False)
    .reset_index(drop=True)
)

display(comparison_df)


In [ ]:
comparison_df.set_index("Model")[
    ["Precision@10", "Recall@10", "nDCG@10"]
].plot(
    kind="bar",
    figsize=(13, 5),
    rot=20,
)

plt.title("Сравнение всех рекомендательных моделей на test")
plt.xlabel("Модель")
plt.ylabel("Значение метрики")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
